In [ ]:
import os
import pandas as pd
import warnings
from datetime import time
warnings.filterwarnings("ignore")

In [ ]:
# Read files
def read_files(folder_path, analyze_contract, analyze_length):
    combined_data = []
    
    csv_files = sorted([file for file in os.listdir(folder_path) if file.endswith('.csv')])
    
    try:
        target_index = csv_files.index(next(file for file in csv_files if analyze_contract in file))
    except StopIteration:
        print(f"{analyze_contract} Not Found!!")
        return None, None
    
    start_index = max(0, target_index - analyze_length)
    selected_files = csv_files[start_index:target_index]
    
    if (len(selected_files) == analyze_length):
        
        for option_file in selected_files:
            file_path = os.path.join(folder_path, option_file)
            ATMS_data = pd.read_csv(file_path)
            
            if 'Unnamed: 0' in ATMS_data.columns:
                ATMS_data = ATMS_data.drop(columns=['Unnamed: 0'])
            
            combined_data.append(ATMS_data)

        history_data = pd.concat(combined_data, ignore_index=True)

        file_path = os.path.join(folder_path, f"{analyze_contract}.csv")
        analyze_data = pd.read_csv(file_path)
        if 'Unnamed: 0' in analyze_data.columns:
                analyze_data = analyze_data.drop(columns=['Unnamed: 0'])

        return history_data, analyze_data
    else:

        return None, None

In [ ]:
# Compute standard deviation
def compute_stats(group):

    atms_values = group['ATMS']
    avg = atms_values.mean()
    
    above = atms_values[atms_values > avg]
    below = atms_values[atms_values < avg]
    
    pos_std = above.std()
    neg_std = below.std()
    
    upbond = avg + 2 * pos_std
    downbond = avg - 2 * neg_std
    
    return pd.Series({
        'ATMS_avg': avg,
        'ATMS_pos_std': pos_std,
        'ATMS_neg_std': neg_std,
        'ATMS_upbond': upbond,
        'ATMS_downbond': downbond
    })

In [ ]:
def is_trading_time(ts):
    # Determines whether the timestamp is within trading hours.

    w = ts.weekday()
    t = ts.time()

    def in_range(t, start, end):
        return (start <= t < end)

    # Wednesday night session (Wed 15:00 - Thu 05:00)
    if (w == 2 and t >= time(15, 0)) or (w == 3 and t < time(5, 0)):
        return True

    # Thursday day session (Thu 08:45 - 13:45)
    if w == 3 and in_range(t, time(8, 45), time(13, 45)):
        return True

    # Thursday night session (Thu 15:00 - Fri 05:00)
    if (w == 3 and t >= time(15, 0)) or (w == 4 and t < time(5, 0)):
        return True

    # Friday day session (Fri 08:45 - 13:45)
    if w == 4 and in_range(t, time(8, 45), time(13, 45)):
        return True

    # Friday night session (Fri 15:00 - Sat 05:00)
    if (w == 4 and t >= time(15, 0)) or (w == 5 and t < time(5, 0)):
        return True

    # Monday day session (Mon 08:45 - 13:45)
    if w == 0 and in_range(t, time(8, 45), time(13, 45)):
        return True

    # Monday night session (Mon 15:00 - Tue 05:00)
    if (w == 0 and t >= time(15, 0)) or (w == 1 and t < time(5, 0)):
        return True

    # Tuesday day session (Tue 08:45 - 13:45)
    if w == 1 and in_range(t, time(8, 45), time(13, 45)):
        return True

    # Tuesday night session (Tue 15:00 - Wed 05:00)
    if (w == 1 and t >= time(15, 0)) or (w == 2 and t < time(5, 0)):
        return True

    # Wednesday day session (Wed 08:45 - 13:30)
    if w == 2 and in_range(t, time(8, 45), time(13, 30)):
        return True

    # All other times are outside of trading hours
    return False

In [ ]:
# Resample
def resample(data, resample_frequency):
    ohlc_cols = ['T_open', 'T_high', 'T_low', 'T_close', 'F_open', 'F_high', 'F_low', 'F_close', 'TX']
    ohlc_resampled = data[ohlc_cols].resample(resample_frequency).agg({
        'T_open': 'first',
        'T_high': 'max',
        'T_low': 'min',
        'T_close': 'last',
        'F_open': 'first',
        'F_high': 'max',
        'F_low': 'min',
        'F_close': 'last',
        'TX': 'first'
    })

    sum_cols = ['F_vol', 'TX_volatility']
    sum_resampled = data[sum_cols].resample(resample_frequency).sum()

    special_cols = [
        'ITM_1_call', 'ITM_1_call_TV', 'ITM_1_put', 'ITM_1_put_TV',
        'OTM_1_call', 'OTM_1_put', 'ATMS', 'ATMS_interpolated',
        'remaining_time', 'time_label', 'ATMS_avg', 'ATMS_pos_std',
        'ATMS_neg_std', 'ATMS_upbond', 'ATMS_downbond'
    ]
    
    special_data = data[special_cols].copy()
    special_data.loc[:, special_cols] = special_data.loc[:, special_cols].fillna(method='bfill')

    special_resampled = special_data.resample(resample_frequency).first()

    resampled_data = pd.concat([ohlc_resampled, sum_resampled, special_resampled], axis=1)
    return resampled_data

In [ ]:
# Calculate deviation
def calculate_ATMS_Nstd(row):
    if (row["ATMS_interpolated"] > row["ATMS_avg"]):
        return (row["ATMS_interpolated"] - row["ATMS_avg"]) /  row["ATMS_pos_std_EMA"]
    else:
        return (row["ATMS_avg"] - row["ATMS_interpolated"]) / row["ATMS_neg_std_EMA"]

In [ ]:
folder_path = "data/ATMS_data"
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]
analyze_length = 12
resample_frequency = "resample_frequency"

In [ ]:
for file_name in csv_files[analyze_length:]:
    analyze_contract = file_name[0:8]

    history_data, analyze_data = read_files(folder_path, analyze_contract, analyze_length)

    df_stats = (history_data.groupby('remaining_time').apply(compute_stats).reset_index())
    
    total_data = analyze_data.merge(df_stats, on="remaining_time", how="left")
    total_data['TX_volatility'] = (total_data['TX'] - total_data['TX'].shift(1)) ** 2

    total_data['timestamp'] = pd.to_datetime(total_data['timestamp'])

    total_data = total_data.set_index('timestamp')

    # Resample
    total_data = resample(total_data, resample_frequency)
    total_data = total_data.reset_index()

    total_data['is_trading'] = total_data['timestamp'].apply(is_trading_time)

    total_data = total_data[total_data['is_trading']].drop(columns=['is_trading'])

    total_data['ATMS_pos_std_EMA'] = total_data['ATMS_pos_std'].ewm(span=analyze_length).mean()
    total_data['ATMS_neg_std_EMA'] = total_data['ATMS_neg_std'].ewm(span=analyze_length).mean()

    total_data["ATMS_Nstd"] = total_data.apply(calculate_ATMS_Nstd, axis=1)

    total_data['time'] = total_data['timestamp'].dt.strftime('%H:%M')

    # Export to CSV
    out_path = f"data/std_data/{analyze_contract}.csv"
    total_data.to_csv(out_path, encoding="utf_8_sig", index=False)